In [7]:
import pandas as pd
df = pd.read_csv("/train.txt", sep=";", names=["text", "emotion"])

print(df.head())



                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     16000 non-null  object
 1   emotion  16000 non-null  object
dtypes: object(2)
memory usage: 250.1+ KB


In [9]:
print(df["emotion"].value_counts())

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64


In [10]:
label2id = {
    "sadness": 0,
    "joy": 1,
    "love": 2,
    "anger": 3,
    "fear": 4,
    "surprise": 5
}

df["label"] = df["emotion"].map(label2id)

In [11]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [12]:
!pip install transformers datasets torch accelerate scikit-learn

In [13]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [14]:
import torch

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [15]:
train_dataset = EmotionDataset(
    train_encodings,
    train_labels
)

val_dataset = EmotionDataset(
    val_encodings,
    val_labels
)

In [16]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.574540,0.174382
2,0.135252,0.163404


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.574540,0.174382
2,0.135252,0.163404
3,0.105843,0.168668


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2400, training_loss=0.2256780687967936, metrics={'train_runtime': 660.0364, 'train_samples_per_second': 58.179, 'train_steps_per_second': 3.636, 'total_flos': 1716861294028800.0, 'train_loss': 0.2256780687967936, 'epoch': 3.0})

In [18]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
0.105843,0.168668,3


{'eval_loss': 0.16866767406463623}

In [19]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(val_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(val_labels, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97       946
           1       0.94      0.96      0.95      1021
           2       0.87      0.84      0.85       296
           3       0.91      0.96      0.93       427
           4       0.89      0.91      0.90       397
           5       0.85      0.75      0.80       113

    accuracy                           0.93      3200
   macro avg       0.91      0.90      0.90      3200
weighted avg       0.93      0.93      0.93      3200



In [20]:
trainer.save_model("emotion_bert")
tokenizer.save_pretrained("emotion_bert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('emotion_bert/tokenizer_config.json', 'emotion_bert/tokenizer.json')

In [33]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch


tokenizer = BertTokenizer.from_pretrained("emotion_bert")
model = BertForSequenceClassification.from_pretrained("emotion_bert")


model.eval()


id2label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}


text = "me and my friend fighted for too long i may not talk to him"


inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)


with torch.no_grad():
    outputs = model(**inputs)


predicted_class = torch.argmax(outputs.logits, dim=1).item()

print("Emotion:", id2label[predicted_class])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Emotion: anger
